# Bake-off candidate: roboflow/sports

Runs the soccer example from https://github.com/roboflow/sports on the canonical
bake-off clip. Outputs trajectories parquet + annotated video.

**Inputs:** `data/bakeoff_clip.mp4`, `data/bakeoff_clip_corners.json` (both from the
soccer-vision repo).

**Outputs:** `data/bakeoff_outputs/roboflow/trajectories.parquet`,
`data/bakeoff_outputs/roboflow/annotated.mp4`.

Run on Colab Pro with T4 or L4 GPU.

In [ ]:
!pip install -q "ultralytics>=8.2" "supervision>=0.22" "git+https://github.com/roboflow/sports.git"
!pip install -q git+https://github.com/PatrickJReed/soccer-vision.git
import os
from pathlib import Path
DATA = Path("data")
DATA.mkdir(exist_ok=True)

In [ ]:
# In Colab, clone the soccer-vision repo for the bake-off clip + corners
!git clone https://github.com/PatrickJReed/soccer-vision.git /content/sv
INPUT_CLIP = Path("/content/sv/data/bakeoff_clip.mp4")
OUT = Path("/content/output/roboflow")
OUT.mkdir(parents=True, exist_ok=True)
assert INPUT_CLIP.exists(), INPUT_CLIP

In [ ]:
# Use the SoccerPitchDetectionModel + SoccerPlayerDetectionModel from sports/examples/soccer
# The exact import paths depend on the repo layout at clone time; check sports/examples/soccer/main.py.
from sports.common.soccer import SoccerPitchDetectionModel, SoccerPlayerDetectionModel
from sports.configs.soccer import SoccerPitchConfiguration
import supervision as sv
import cv2

CONFIG = SoccerPitchConfiguration()
player_model = SoccerPlayerDetectionModel()  # uses default weights
pitch_model = SoccerPitchDetectionModel()
byte_tracker = sv.ByteTrack()

cap = cv2.VideoCapture(str(INPUT_CLIP))
fps = cap.get(cv2.CAP_PROP_FPS)
records: list[dict] = []
frame_idx = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    # Player detections
    player_dets = player_model.infer(frame)
    tracked = byte_tracker.update_with_detections(player_dets)

    # Each detection becomes a row
    for box, tid, cls, conf in zip(
        tracked.xyxy, tracked.tracker_id, tracked.class_id, tracked.confidence, strict=False
    ):
        if tid is None:
            continue
        x1, y1, x2, y2 = box.tolist()
        cx, cy = (x1 + x2) / 2, y2  # foot position
        cls_name = {0: "player", 1: "goalkeeper", 2: "referee", 3: "ball"}.get(int(cls), "player")
        records.append({
            "frame": frame_idx,
            "t_seconds": frame_idx / fps,
            "track_id": int(tid),
            "x_px": cx,
            "y_px": cy,
            "bbox_x1": x1, "bbox_y1": y1, "bbox_x2": x2, "bbox_y2": y2,
            "class": cls_name,
            "team": "unknown",   # team classification done in Cell 5
            "conf": float(conf),
        })
    frame_idx += 1

cap.release()
print(f"Processed {frame_idx} frames; {len(records)} detection rows.")

In [ ]:
# Sketch — actual code follows roboflow/sports/examples/soccer/team.py
# This cell crops each player box, embeds with SigLIP, KMeans into 2 clusters,
# assigns 'own' (cluster 0) vs 'opp' (cluster 1) — arbitrary which is which for now.
import pandas as pd
df = pd.DataFrame(records)
# TODO when running: apply SigLIP team classifier from roboflow.sports.examples.soccer.team
# TODO when running: update df["team"] with "own"/"opp" assignments per row
# For the bake-off scoring, we only need 2 stable clusters; which is "own" is fixed later by inspection.

In [ ]:
from soccer_vision.io.schema import validate_trajectories
df = df.astype({"frame": "int64", "track_id": "int64"})
validate_trajectories(df)
df.to_parquet(OUT / "trajectories.parquet")
print(f"Wrote {OUT / 'trajectories.parquet'}: {len(df)} rows")

In [ ]:
# Render boxes + IDs onto the clip for visual review in the synthesis notebook
import supervision as sv

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

cap = cv2.VideoCapture(str(INPUT_CLIP))
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(str(OUT / "annotated.mp4"),
                         cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Group by frame, render each
for frame_idx, group in df.groupby("frame"):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        break
    # TODO when running: build detections from rows and annotate (full code: use sv.Detections)
    # TODO when running: render boxes + labels onto the frame
    writer.write(frame)

writer.release()
cap.release()
print(f"Wrote {OUT / 'annotated.mp4'}")

In [ ]:
# Download these files from Colab and commit them at:
#   data/bakeoff_outputs/roboflow/trajectories.parquet
#   data/bakeoff_outputs/roboflow/annotated.mp4
from google.colab import files
files.download(str(OUT / "trajectories.parquet"))
files.download(str(OUT / "annotated.mp4"))